# CUT + SC Loss 재도전 — 제안서 핵심 기여 재검증 (직접 실행)

제안서의 **Structural Consistency Loss**(구조 일관성 손실, `L_total = L_CUT + λ·L_SC`)가 이전 λ=1.0에서 **전 지표 악화**로 실패했음.
가설: λ가 너무 커서 출력을 밋밋한 입력으로 끌어당김. → **λ를 낮춰(0.1) HAT 도메인에서 재검증**, plain CUT(챔피언)과 직접 비교.

- 커널 `trellis` → Restart Kernel → Run All
- `LAMBDA_SC`·`SC_BACKEND`(dinov3|vgg) 조절 → **λ 스윕이 이 실험의 핵심** (0.01 / 0.1 / 0.5)
- 학습 35ep ≈ **1.5~2h** (빨리 보려면 N_EPOCHS 줄이기). 자동 저장.
- 기준(480장): CycleGAN FID 198.6 / **plain CUT FID 171.8**(현 챔피언)
- ⚠️ **육안 검증 필수**(셀 5) — 지표가 반복적으로 속였음([[교훈]])


## 0. 환경 + 설정 + 로더

In [ ]:
import os, sys, time, glob
import numpy as np
from PIL import Image
import torch
import matplotlib.pyplot as plt
ROOT = os.path.expanduser("~/erang_sr")
CUT  = os.path.join(ROOT, "refiners/cut_src")
os.chdir(CUT); sys.path.insert(0, CUT)
for p in (ROOT, os.path.join(ROOT, "team_aiduo")):
    if p not in sys.path: sys.path.append(p)
DEV="cuda"

# ===== 설정 =====
LAMBDA_SC  = 0.1        # ★ SC 가중치 (이전 실패는 1.0). 0.01/0.1/0.5 스윕해볼 것
SC_BACKEND = "dinov3"   # dinov3 | vgg  (제안서는 dinov3)
N_EPOCHS = 25           # (빨리 보려면 5)
N_DECAY  = 10           # (빨리 보려면 0)
SAVE_EVERY = 5
DR = os.path.join(ROOT, "refiners/data/refine_hat")
EVAL = os.path.join(ROOT, "runs/eval_run")   # plain_cut/cyclegan/sr_hat/hr_ref pngs

def load01(p):
    im=np.asarray(Image.open(p).convert("RGB")).astype(np.float32)/255.0
    return torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).float()
def save01(t,p):
    a=(t.detach().cpu().clamp(0,1).numpy().transpose(1,2,0)*255).round().astype("uint8"); Image.fromarray(a).save(p)
def load_G(ckpt):
    from models import networks
    net=networks.define_G(3,3,64,"resnet_9blocks","instance",False,"xavier",0.02,False,False,[0])
    sd=torch.load(ckpt, map_location=DEV)
    (net.module if hasattr(net,"module") else net).load_state_dict(sd); net.eval(); return net
def refine(G,x):
    with torch.no_grad(): return ((G(x*2-1)+1)/2).clamp(0,1)
SC_NAME=f"cutsc_{SC_BACKEND}_l{int(round(LAMBDA_SC*100)):03d}"
print("device:", torch.cuda.get_device_name(0), "| SC:", SC_BACKEND, "λ=", LAMBDA_SC, "| name:", SC_NAME)


## 1. 데이터 + 비교 pngs 확인

In [ ]:
for s in ["trainA","trainB","testA","testB"]:
    print(f"  {s}: {len(os.listdir(os.path.join(DR,s)))}장")
for d in ["sr_hat","cyclegan","plain_cut","hr_ref"]:
    p=os.path.join(EVAL,d); print(f"  ref {d}: {len(os.listdir(p)) if os.path.isdir(p) else 0}장",
          "" if os.path.isdir(p) and len(os.listdir(p))>=480 else " <- 없으면 cut_res_run_experiment 셀7 먼저")
print("plain CUT 가중치:", os.path.exists(os.path.join(CUT,"checkpoints/cut_plain_full/final_net_G.pth")))


## 2. CUT+SC 학습 (loss·SC 관찰 + 저장)
`--model cutsc --lambda_SC LAMBDA_SC --sc_backend SC_BACKEND`. SC loss가 함께 흐름.

In [ ]:
sys.argv = ["train.py","--dataroot",DR,"--name",SC_NAME,"--model","cutsc","--CUT_mode","CUT",
    "--lambda_SC",str(LAMBDA_SC),"--sc_backend",SC_BACKEND,
    "--display_id","0","--gpu_ids","0","--batch_size","1",
    "--n_epochs",str(N_EPOCHS),"--n_epochs_decay",str(N_DECAY),
    "--load_size","512","--crop_size","256","--print_freq","200"]
from options.train_options import TrainOptions
from data import create_dataset
from models import create_model
opt = TrainOptions().parse(); opt.num_threads=0
dataset=create_dataset(opt); model=create_model(opt)
total=opt.n_epochs+opt.n_epochs_decay
print(f"\n[{SC_NAME}] {len(dataset)}장 · {total} epoch (SC={SC_BACKEND}, λ={LAMBDA_SC})\n")
t0=time.time(); step=0
for epoch in range(opt.epoch_count, total+1):
    for i,data in enumerate(dataset):
        if epoch==opt.epoch_count and i==0:
            model.data_dependent_initialize(data); model.setup(opt); model.parallelize()
        model.set_input(data); model.optimize_parameters(); step+=1
        if step%100==0:
            L=model.get_current_losses()
            print(f"  ep{epoch} step{step:>5}  G_GAN {L['G_GAN']:.3f}  NCE {L['NCE']:.3f}  SC {L.get('SC',0):.3f}  ({time.time()-t0:.0f}s)")
    if epoch%SAVE_EVERY==0 or epoch==total:
        model.save_networks("latest"); print(f"  --- ep{epoch} 저장 ({time.time()-t0:.0f}s) ---")
model.save_networks("final")
print(f"[{SC_NAME}] 완료·저장 → checkpoints/{SC_NAME}/final_net_G.pth ({time.time()-t0:.0f}s)")


## 3. refined 결과 (테스트 3장): 입력 / plain CUT / CUT+SC / HR

In [ ]:
G_sc = model.netG; G_sc.eval()
G_plain = load_G(os.path.join(CUT,"checkpoints/cut_plain_full/final_net_G.pth"))
tests=sorted(glob.glob(DR+"/testA/*.png"))[:3]
cols=["input","plain CUT",f"CUT+SC({SC_BACKEND} λ{LAMBDA_SC})","HR (GT)"]
fig,ax=plt.subplots(len(tests),4,figsize=(14,3.6*len(tests)))
for r,ta in enumerate(tests):
    x=load01(ta).to(DEV)
    imgs=[x[0].cpu(), refine(G_plain,x)[0].cpu(), refine(G_sc,x)[0].cpu(), load01(ta.replace("testA","testB"))[0]]
    for c,(im,t) in enumerate(zip(imgs,cols)):
        ax[r,c].imshow(im.numpy().transpose(1,2,0)); ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()


## 4. 480장 평가 — plain CUT vs CUT+SC (환각=FID 개선됐나)

In [ ]:
from cycleGen_model import load_cyclegan_model
G_cyc=load_cyclegan_model(os.path.join(ROOT,"weights/best_G_AB.pth"), device=DEV)
allA=sorted(glob.glob(DR+"/trainA/*.png"))+sorted(glob.glob(DR+"/testA/*.png"))
test_names=set(os.path.basename(p) for p in glob.glob(DR+"/testA/*.png"))
base=os.path.join(ROOT,"runs/eval_sc"); scdir=os.path.join(base,"cutsc"); hr_ref=os.path.join(EVAL,"hr_ref")
os.makedirs(scdir,exist_ok=True)
print(f"CUT+SC {len(allA)}장 생성...")
for j,ta in enumerate(allA):
    save01(refine(G_sc, load01(ta).to(DEV))[0], os.path.join(scdir, os.path.basename(ta)))
    if (j+1)%150==0: print(f"  {j+1}/{len(allA)}")
print("지표 계산...")
from skimage.metrics import structural_similarity as ssim
import lpips as _L, pyiqa
from pytorch_fid.fid_score import calculate_fid_given_paths
lp=_L.LPIPS(net="alex").to(DEV); niqe=pyiqa.create_metric("niqe",device=DEV)
def psnr(a,b):
    m=float(((a-b)**2).mean()); return 99.0 if m==0 else 10*np.log10(1/m)
conds={"cyclegan":os.path.join(EVAL,"cyclegan"),"plain_cut":os.path.join(EVAL,"plain_cut"),SC_NAME:scdir}
rows={}
for cond,dd in conds.items():
    ps,ss,lps=[],[],[]
    for n in sorted(test_names):
        out=load01(os.path.join(dd,n)); hr=load01(os.path.join(hr_ref,n))
        a=out[0].numpy().transpose(1,2,0); b=hr[0].numpy().transpose(1,2,0)
        ps.append(psnr(a,b)); ss.append(ssim(a,b,channel_axis=2,data_range=1.0))
        with torch.no_grad(): lps.append(float(lp(out.to(DEV)*2-1,hr.to(DEV)*2-1).mean()))
    nqs=[float(niqe(os.path.join(dd,x))) for x in os.listdir(dd)]
    fid=calculate_fid_given_paths([dd,hr_ref],batch_size=50,device=DEV,dims=2048)
    rows[cond]=(np.mean(ps),np.mean(ss),np.mean(lps),fid,np.mean(nqs))
print(f"\n===== 480장 (HAT, SC={SC_BACKEND} λ={LAMBDA_SC}) =====")
print(f"{'조건':16s} | {'PSNR up':>8} {'SSIM up':>8} {'LPIPS dn':>9} | {'FID dn':>8} {'NIQE dn':>8}")
print("-"*66)
for k in ["cyclegan","plain_cut",SC_NAME]:
    p,s,l,f,n=rows[k]; print(f"{k:16s} | {p:8.3f} {s:8.4f} {l:9.4f} | {f:8.3f} {n:8.3f}")
print("\n판정: CUT+SC의 FID가 plain CUT(171.8)보다 낮고 육안도 정상이면 → SC 살아남(제안서 기여 확정). 아니면 plain CUT 유지.")


## 5. 육안 5-way (필수)

In [ ]:
names=sorted(test_names)[:3]
srcs=[("input","sr_hat"),("CycleGAN","cyclegan"),("plain CUT","plain_cut"),(f"CUT+SC",None),("HR (GT)","hr_ref")]
fig,ax=plt.subplots(3,5,figsize=(18,10.5))
for r,n in enumerate(names):
    imgs=[]
    for t,d in srcs:
        if d is None: imgs.append(load01(os.path.join(scdir,n))[0])
        else: imgs.append(load01(os.path.join(EVAL,d,n))[0])
    for c,(im,(t,_)) in enumerate(zip(imgs,srcs)):
        ax[r,c].imshow(im.numpy().transpose(1,2,0)); ax[r,c].axis("off")
        if r==0: ax[r,c].set_title(t)
plt.tight_layout(); plt.show()
print("CUT+SC가 plain CUT보다 (a)환각 더 적고 (b)선명 유지 (c)색 정상 → SC 유효. 흐려지거나 나빠지면 → SC 무효, plain CUT.")
